### ⚠️ Importará toda a carga de novo!

In [0]:
import requests
import json
import pandas as pd

In [0]:
tmdb_key = dbutils.secrets.get(scope='tmdb_keys', key='tmdb_key')

In [0]:
#Criando a base de dados de filmes recomendáveis chamando a API do TMDB

url = "https://api.themoviedb.org/3/discover/movie"

headers = {
    "accept": "application/json",
    "Authorization": tmdb_key
}

movies = []

#Dew ano em ano
for year in range(1980, 2027):

    #De pagina em pagina
    for page in range(1, 501):
        params = {
            "primary_release_year": year,
            "vote_count.gte": 200,
            "sort_by": "popularity.desc",
            "page": page,
        }

        #
        response = requests.get(url, headers= headers, params=params)
        data = response.json()

        #para quando nao tem mais paginas naquele ano
        if len(data['results']) == 0:
            break
        
        #adiciona ao array
        for movie in data['results']:
            movies.append({
                "id": movie["id"],
                 "title": movie["title"],
                "overview": movie["overview"],
                "genre_ids": movie["genre_ids"],
                })

df = pd.DataFrame(movies)


In [0]:
headers = {
    "accept": "application/json",
    "Authorization": tmdb_key
}

keywords_list = []

session = requests.Session()

for movie_id in df['id']:
    url = f"https://api.themoviedb.org/3/movie/{movie_id}/keywords"
    response_keywords = session.get(url, headers=headers)
    movie_keyword = response_keywords.json()

    keywords = []

    for keyword in movie_keyword['keywords']:
        keywords.append(keyword['name'])

    keywords_list.append(" ".join(keywords))

df['keywords'] = keywords_list

In [0]:
spark_df = spark.createDataFrame(df)
spark_df.write.mode('overwrite').option("mergeSchema", "true").saveAsTable('workspace.bronze.recomendation_db_bronze')